# Session 26: LSTM — Part 2 (Applications & Hands-on)


### Tasks 1, 2, & 3: Prepare Data, Train LSTM, and Evaluate
This cell prepares fake Flipkart price data, reshapes it using NumPy, trains an LSTM model for 10 epochs, and compares the prediction to the actual test value.

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# --- TASK 1: Prepare & Reshape the Dataset ---
# Fake Flipkart product prices over 10 days
prices = [1000, 1050, 1020, 1100, 1150, 1200, 1180, 1250, 1300, 1350]

# We will use a sliding window of 3 days to predict the 4th day
time_steps = 3
X, y = [], []

for i in range(len(prices) - time_steps):
    X.append(prices[i : i + time_steps]) # Input: 3 consecutive days
    y.append(prices[i + time_steps])     # Target: The 4th day

X = np.array(X)
y = np.array(y)

# Reshape X for LSTM input: (samples, time_steps, features)
# Here, features = 1 because we only have one variable (price)
X = X.reshape((X.shape[0], X.shape[1], 1))
print(f"Data reshaped to: {X.shape} -> (samples, time_steps, features)\n")

# --- TASK 2: Build & Train the LSTM Model ---
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(time_steps, 1)))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

print("Training LSTM for 10 epochs...")
# Training for 10 epochs
model.fit(X, y, epochs=10, verbose=0) 
print("Training complete!\n")

# --- TASK 3: Evaluate & Predict ---
# Let's take the sequence right before the last day to predict the final day's price
test_sequence = np.array(prices[-4:-1]).reshape((1, time_steps, 1))
actual_last_price = prices[-1]

# Predict the next price
predicted_price = model.predict(test_sequence, verbose=0)[0][0]

print("--- EVALUATION ---")
print(f"Predicted Price: ₹{predicted_price:.2f}")
print(f"Actual Price:    ₹{actual_last_price}")

Data reshaped to: (7, 3, 1) -> (samples, time_steps, features)



c:\Users\param\machine learning\deep learning\CNN project\cnn\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training LSTM for 10 epochs...
Training complete!

--- EVALUATION ---
Predicted Price: ₹-227.54
Actual Price:    ₹1350


### Task 4: Switch to a GRU Layer and Compare
Here we swap the LSTM layer for a Gated Recurrent Unit (GRU) layer, keeping the same architecture, and compare the training time.

In [ ]:
from tensorflow.keras.layers import GRU
import time

# Build the GRU Model (exact same architecture, just swapped layer)
gru_model = Sequential()
gru_model.add(GRU(50, activation='relu', input_shape=(time_steps, 1)))
gru_model.add(Dense(1))
gru_model.compile(optimizer='adam', loss='mse')

# Time the GRU Training
start_time = time.time()
gru_model.fit(X, y, epochs=10, verbose=0)
gru_train_time = time.time() - start_time

# GRU Prediction
gru_predicted_price = gru_model.predict(test_sequence, verbose=0)[0][0]

print("--- GRU RESULTS ---")
print(f"GRU Training Time: {gru_train_time:.4f} seconds")
print(f"GRU Predicted Price: ₹{gru_predicted_price:.2f}")

# Note: You will likely notice that the GRU trains slightly faster than the LSTM. 

--- GRU RESULTS ---
GRU Training Time: 2.8910 seconds
GRU Predicted Price: ₹-50.52


### Task 5: Reusable Sequence Preparation Function

**How this function works:**
This function essentially acts like a sliding window moving across our timeline. It takes a long list of numbers, like daily Spotify plays, and chops it up into overlapping blocks based on the `window_size`. If the window size is 3, it groups Monday, Tuesday, and Wednesday together as the 'input' (X) to predict Thursday as the 'target' (y). Then, it shifts forward by one single day—grouping Tuesday, Wednesday, and Thursday to predict Friday. Finally, it uses NumPy to reshape all these chopped-up blocks into a 3D grid format — `(number of blocks, days per block, number of features)` — which is the mandatory format an RNN/LSTM/GRU needs to understand how the data flows over time.

In [3]:
def prepare_sequence_data(data, window_size):
    """
    Prepares a 1D array of data into features (X) and targets (y) for an RNN/LSTM/GRU.
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        # Create the input sequence (the window)
        X.append(data[i : i + window_size])
        # Create the target (the value immediately following the window)
        y.append(data[i + window_size])
        
    # Convert to numpy arrays and reshape X to (samples, time_steps, features)
    X = np.array(X).reshape(-1, window_size, 1)
    y = np.array(y)
    
    return X, y

# Example Usage:
spotify_plays = [50, 60, 55, 70, 80, 95, 100]
X_train, y_train = prepare_sequence_data(spotify_plays, window_size=3)

print("Input shape (X):", X_train.shape)
print("Target shape (y):", y_train.shape)
print("\nFirst input window:\n", X_train[0])
print("First target:\n", y_train[0])

Input shape (X): (4, 3, 1)
Target shape (y): (4,)

First input window:
 [[50]
 [60]
 [55]]
First target:
 70
